RNA + Protein + CNV is worth testing because each modality showed predictive signal alone.
But improvement is not guaranteed.

Why it may help ?

RNA = transcription signal
Protein = pathway/activity signal
CNV = DNA structural signal

Why it may not help ?

too many features
small dataset
redundant information
added noise

In [ ]:
# ==============================
# 1. Import libraries
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# ==============================
# 2. Load dataset
# ==============================

df = pd.read_csv("/content/brca_data_w_subtypes.csv")

print(df.shape)
df.head()

(705, 1941)


,rs_CLEC3A,rs_CPB1,rs_SCGB2A2,rs_SCGB1D2,rs_TFF1,rs_MUCL1,rs_GSTM1,rs_PIP,rs_ADIPOQ,rs_ADH1B,...,pp_p62.LCK.ligand,pp_p70S6K,pp_p70S6K.pT389,pp_p90RSK,pp_p90RSK.pT359.S363,vital.status,PR.Status,ER.Status,HER2.Final.Status,histological.type
0,0.892818,6.580103,14.123672,10.606501,13.189237,6.649466,10.520335,10.338490,10.248379,10.229970,...,-0.691766,-0.337863,-0.178503,0.011638,-0.207257,0,Positive,Positive,Negative,infiltrating ductal carcinoma
1,0.000000,3.691311,17.116090,15.517231,9.867616,9.691667,8.179522,7.911723,1.289598,1.818891,...,0.279067,0.292925,-0.155242,-0.089365,0.267530,0,Positive,Negative,Negative,infiltrating ductal carcinoma
2,3.748150,4.375255,9.658123,5.326983,12.109539,11.644307,10.517330,5.114925,11.975349,11.911437,...,0.219910,0.308110,-0.190794,-0.222150,-0.198518,0,Positive,Positive,Negative,infiltrating ductal carcinoma
3,0.000000,18.235519,18.535480,14.533584,14.078992,8.913760,10.557465,13.304434,8.205059,9.211476,...,-0.266554,-0.079871,-0.463237,0.522998,-0.046902,0,Positive,Positive,Negative,infiltrating ductal carcinoma
4,0.000000,4.583724,15.711865,12.804521,8.881669,8.430028,12.964607,6.806517,4.294341,5.385714,...,-0.441542,-0.152317,0.511386,-0.096482,0.037473,0,Positive,Positive,Negative,infiltrating ductal carcinoma


ER status is biologically connected to hormone signaling, transcriptional activity, and protein-level pathway activity. Protein data may add information that RNA alone does not fully capture.

Mutation data can be useful, but for ER-status prediction it may be weaker because ER status is often more expression/pathway-driven than mutation-driven.

We compared RNA-only models with RNA-integrated multi-omics models by adding protein expression, mutation, and copy-number features to evaluate whether additional molecular layers improve ER-status prediction.

In [ ]:
# ==============================
# 3. Select features and target
# ==============================
copy_cols_rs = [col for col in df.columns if col.startswith("rs_")]

copy_cols_cn = [col for col in df.columns if col.startswith("cn_")]


copy_cols_pp = [col for col in df.columns if col.startswith("pp_")]

copy_cols = copy_cols_rs + copy_cols_cn + copy_cols_pp

X = df[copy_cols]
y = df["ER.Status"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (705, 1687)
Target shape: (705,)


In [ ]:
# ==============================
# 4. Check target values before cleaning
# ==============================

print(y.value_counts(dropna=False))

ER.Status
Positive                       414
Negative                       135
NaN                            122
Not Performed                   27
Performed but Not Available      5
Indeterminate                    2
Name: count, dtype: int64


In [ ]:
# ==============================
# 5. Clean target labels
# Remove NaN, Indeterminate, Not Performed, etc.
# ==============================

valid_classes = ["Positive", "Negative"]

valid_rows = (
    y.notna() &
    y.isin(valid_classes)
)

X_clean = X.loc[valid_rows].copy()
y_clean = y.loc[valid_rows].copy()

print("Original rows:", len(y))
print("Cleaned rows:", len(y_clean))
print("Removed rows:", len(y) - len(y_clean))

print("\nCleaned target counts:")
print(y_clean.value_counts())

print("\nCleaned target proportions:")
print(y_clean.value_counts(normalize=True))

Original rows: 705
Cleaned rows: 549
Removed rows: 156

Cleaned target counts:
ER.Status
Positive    414
Negative    135
Name: count, dtype: int64

Cleaned target proportions:
ER.Status
Positive    0.754098
Negative    0.245902
Name: proportion, dtype: float64


In [ ]:
# ==============================
# 6. Encode labels
# ==============================

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_clean)

print("Label mapping:")
for label, code in zip(encoder.classes_, encoder.transform(encoder.classes_)):
    print(label, "=", code)

Label mapping:
Negative = 0
Positive = 1


In [ ]:
# ==============================
# 7. Train-test split
# Stratify keeps the same class ratio
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (439, 1687)
X_test: (110, 1687)
y_train: (439,)
y_test: (110,)


In [ ]:
# ==============================
# 8. Check class balance after split
# ==============================

print("Training class distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

Training class distribution:
1    0.753986
0    0.246014
Name: proportion, dtype: float64

Testing class distribution:
1    0.754545
0    0.245455
Name: proportion, dtype: float64


In [ ]:
# ==============================
# 9. Feature selection
# IMPORTANT: fit only on training data
# ==============================

selector = SelectKBest(score_func=f_classif, k=300)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)



print("Selected train shape:", X_train_selected.shape)
print("Selected test shape:", X_test_selected.shape)

Selected train shape: (439, 300)
Selected test shape: (110, 300)


In [ ]:
# ==============================
# 10. Train model
# class_weight helps with imbalance
# ==============================

model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score
)

# ==============================
# 1. Predictions
# ==============================

y_pred = model.predict(X_test_selected)

# predicted probabilities
y_proba = model.predict_proba(X_test_selected)[:, 1]

In [ ]:
# ==============================
# 2. Basic evaluation
# ==============================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

Confusion Matrix:
[[21  6]
 [ 5 78]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.78      0.79        27
    Positive       0.93      0.94      0.93        83

    accuracy                           0.90       110
   macro avg       0.87      0.86      0.86       110
weighted avg       0.90      0.90      0.90       110



In [ ]:
# ==============================
# 3. Individual metrics
# ==============================

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_proba)

print("Accuracy:", accuracy)
print("Balanced Accuracy:", balanced_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.9
Balanced Accuracy: 0.8587684069611781
Precision: 0.9285714285714286
Recall: 0.9397590361445783
F1-score: 0.9341317365269461
ROC-AUC: 0.929941990182954


In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)
rf_model.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=3, min_samples_split=5,
                       n_estimators=300, random_state=42)

In [ ]:
y_pred = rf_model.predict(X_test_selected)
y_proba = rf_model.predict_proba(X_test_selected)[:, 1]
print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[21  6]
 [ 5 78]]
              precision    recall  f1-score   support

    Negative       0.81      0.78      0.79        27
    Positive       0.93      0.94      0.93        83

    accuracy                           0.90       110
   macro avg       0.87      0.86      0.86       110
weighted avg       0.90      0.90      0.90       110

Accuracy: 0.9
Balanced Accuracy: 0.8587684069611781
Precision: 0.9285714285714286
Recall: 0.9397590361445783
F1-score: 0.9341317365269461
ROC-AUC: 0.9370816599732262


In [ ]:
import numpy as np

negative_count = np.sum(y_train == 0)
positive_count = np.sum(y_train == 1)

scale_pos_weight = negative_count / positive_count

print(scale_pos_weight)

0.32628398791540786


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_selected, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = xgb_model.predict(X_test_selected)
y_proba = xgb_model.predict_proba(X_test_selected)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[22  5]
 [ 5 78]]
              precision    recall  f1-score   support

    Negative       0.81      0.81      0.81        27
    Positive       0.94      0.94      0.94        83

    accuracy                           0.91       110
   macro avg       0.88      0.88      0.88       110
weighted avg       0.91      0.91      0.91       110

Accuracy: 0.9090909090909091
Balanced Accuracy: 0.8772869254796966
Precision: 0.9397590361445783
Recall: 0.9397590361445783
F1-score: 0.9397590361445783
ROC-AUC: 0.9259259259259259


The selected XGBoost hyperparameters were chosen to create a balanced and stable model for a high-dimensional genomics dataset with a relatively small sample size and class imbalance.



A smaller tree depth was used to reduce overfitting and improve generalization since deeper trees can memorize noisy gene expression patterns instead of learning meaningful biological relationships.




A lower learning rate was selected to allow the model to learn gradually and more stably rather than making overly aggressive updates. The number of estimators was kept moderate to provide sufficient learning capacity while avoiding excessive model complexity.
 A higher minimum child weight was used to prevent unreliable splits formed from very small subsets of samples, which is important in genomic datasets where noise can easily influence tree construction. Subsampling was applied so that each tree learns from only a portion of the training samples, improving robustness and reducing overfitting.



 Column sampling was also used so that each tree considers only a subset of gene features, which is especially useful in high-dimensional biological data containing many potentially noisy or redundant variables.



 Additionally, scale_pos_weight was included to address class imbalance by giving greater importance to the minority class during training, helping the model avoid bias toward the majority class.

In [ ]:
from xgboost import XGBClassifier

xgb_model_changed = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.5,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_model_changed.fit(X_train_selected, y_train)
y_pred = xgb_model_changed.predict(X_test_selected)
y_proba = xgb_model_changed.predict_proba(X_test_selected)[:, 1]
print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[23  4]
 [ 5 78]]
              precision    recall  f1-score   support

    Negative       0.82      0.85      0.84        27
    Positive       0.95      0.94      0.95        83

    accuracy                           0.92       110
   macro avg       0.89      0.90      0.89       110
weighted avg       0.92      0.92      0.92       110

Accuracy: 0.9181818181818182
Balanced Accuracy: 0.895805443998215
Precision: 0.9512195121951219
Recall: 0.9397590361445783
F1-score: 0.9454545454545454
ROC-AUC: 0.9344042838018741


In [ ]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

The RBF kernel allows SVM to capture nonlinear relationships between genes and ER status.

The balanced class weight helps the model pay more attention to the minority class during training without changing dataset size

A moderate C value helps prevent overfitting while still allowing the model to learn meaningful class boundaries.

Gamma set to scale automatically adapts to the feature variance and is usually a safe starting point for high-dimensional biological data

In [ ]:
svm_model.fit(X_train_selected, y_train)
y_pred = svm_model.predict(X_test_selected)

y_proba = svm_model.predict_proba(X_test_selected)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[21  6]
 [ 6 77]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.78      0.78      0.78        27
    Positive       0.93      0.93      0.93        83

    accuracy                           0.89       110
   macro avg       0.85      0.85      0.85       110
weighted avg       0.89      0.89      0.89       110

Accuracy: 0.8909090909090909
Balanced Accuracy: 0.8527443105756358
Precision: 0.927710843373494
Recall: 0.927710843373494
F1-score: 0.927710843373494
ROC-AUC: 0.9098616688978135


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

In [ ]:
svm_model.fit(X_train_scaled, y_train)
y_pred = svm_model.predict(X_test_scaled)

y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [ 5 78]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.85      0.84        27
    Positive       0.95      0.94      0.95        83

    accuracy                           0.92       110
   macro avg       0.89      0.90      0.89       110
weighted avg       0.92      0.92      0.92       110

Accuracy: 0.9181818181818182
Balanced Accuracy: 0.895805443998215
Precision: 0.9512195121951219
Recall: 0.9397590361445783
F1-score: 0.9454545454545454
ROC-AUC: 0.9004908522980812


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

In [ ]:
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42)

In [ ]:
y_pred = lr_model.predict(X_test_scaled)

y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [ 8 75]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.85      0.79        27
    Positive       0.95      0.90      0.93        83

    accuracy                           0.89       110
   macro avg       0.85      0.88      0.86       110
weighted avg       0.90      0.89      0.89       110

Accuracy: 0.8909090909090909
Balanced Accuracy: 0.8777331548415885
Precision: 0.9493670886075949
Recall: 0.9036144578313253
F1-score: 0.9259259259259259
ROC-AUC: 0.9152164212405176


In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=7,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.5,
    class_weight="balanced",
    random_state=42
)

lgbm_model.fit(X_train_selected, y_train)

y_pred = lgbm_model.predict(X_test_selected)
y_proba = lgbm_model.predict_proba(X_test_selected)[:, 1]

[LightGBM] [Info] Number of positive: 331, number of negative: 108
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002670 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 31692
[LightGBM] [Info] Number of data points in the train set: 439, number of used features: 300
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [ 6 77]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.79      0.85      0.82        27
    Positive       0.95      0.93      0.94        83

    accuracy                           0.91       110
   macro avg       0.87      0.89      0.88       110
weighted avg       0.91      0.91      0.91       110

Accuracy: 0.9090909090909091
Balanced Accuracy: 0.8897813476126729
Precision: 0.9506172839506173
Recall: 0.927710843373494
F1-score: 0.9390243902439024
ROC-AUC: 0.9165551093261936


Scientifically this is actually a strong result
**tree ensemble methods dominate linear models**
We are discovering that:


for this ER genomics problem

which is biologically plausible because gene interactions are often nonlinear.

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=5,
    auto_class_weights="Balanced",
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)

cat_model.fit(X_train_selected, y_train)

y_pred = cat_model.predict(X_test_selected)
y_proba = cat_model.predict_proba(X_test_selected)[:, 1]

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [ 5 78]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.85      0.84        27
    Positive       0.95      0.94      0.95        83

    accuracy                           0.92       110
   macro avg       0.89      0.90      0.89       110
weighted avg       0.92      0.92      0.92       110

Accuracy: 0.9181818181818182
Balanced Accuracy: 0.895805443998215
Precision: 0.9512195121951219
Recall: 0.9397590361445783
F1-score: 0.9454545454545454
ROC-AUC: 0.9361892012494422


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def run_cv_for_model(model_name, model, X_clean, y_encoded, cv, selector):

    scaling_models = ["SVM", "Logistic Regression"]

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X_clean, y_encoded), 1):

        print("Fold:", fold)

        X_train = X_clean.iloc[train_idx]
        X_test = X_clean.iloc[test_idx]

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        print("Train shape:", X_train.shape)
        print("Test shape:", X_test.shape)

        fold_selector = clone(selector)

        X_train_selected = fold_selector.fit_transform(X_train, y_train)
        X_test_selected = fold_selector.transform(X_test)

        selected_features = X_train.columns[fold_selector.get_support()]

        print("\nSelected Features:")
        print(selected_features.tolist())
        print(len(selected_features.tolist()))
        print(len(X_train_selected[1].tolist()))


        if model_name in scaling_models:
            scaler = StandardScaler()
            X_train_selected = scaler.fit_transform(X_train_selected)
            X_test_selected = scaler.transform(X_test_selected)

        fold_model = clone(model)

        fold_model.fit(X_train_selected, y_train)

        y_pred = fold_model.predict(X_test_selected)
        y_proba = fold_model.predict_proba(X_test_selected)[:, 1]

        fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1-score": f1_score(y_test, y_pred),
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            # Weighted metrics
            "Weighted Precision": precision_score(y_test, y_pred, average="weighted"),
            "Weighted Recall": recall_score(y_test, y_pred, average="weighted"),
            "Weighted F1-score": f1_score(y_test, y_pred, average="weighted"),

            # Macro metrics
            "Macro Precision": precision_score(y_test, y_pred, average="macro"),
            "Macro Recall": recall_score(y_test, y_pred, average="macro"),
            "Macro F1-score": f1_score(y_test, y_pred, average="macro"),

            "ROC-AUC": roc_auc_score(y_test, y_proba)
})



    return pd.DataFrame(fold_results)

In [ ]:
rf_cv = run_cv_for_model("Random Forest", rf_model, X_clean, y_encoded, cv, selector)

xgb_cv = run_cv_for_model("XGBoost", xgb_model, X_clean, y_encoded, cv, selector)

lgbm_cv = run_cv_for_model("LightGBM", lgbm_model, X_clean, y_encoded, cv, selector)

cat_cv = run_cv_for_model("CatBoost", cat_model, X_clean, y_encoded, cv, selector)

svm_cv = run_cv_for_model("SVM", svm_model, X_clean, y_encoded, cv, selector)

lr_cv = run_cv_for_model("Logistic Regression", lr_model, X_clean, y_encoded, cv, selector)

Fold: 1
Train shape: (439, 1687)
Test shape: (110, 1687)

Selected Features:
['rs_CPB1', 'rs_TFF1', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_SERPINA6', 'rs_AGR3', 'rs_DHRS2', 'rs_KCNJ3', 'rs_GABRP', 'rs_SLC30A8', 'rs_STAC2', 'rs_CEACAM5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_CST9', 'rs_TFF3', 'rs_BMPR1B', 'rs_AGR2', 'rs_NPY1R', 'rs_C20orf114', 'rs_KIF1A', 'rs_PROM1', 'rs_ELF5', 'rs_GFRA1', 'rs_PGR', 'rs_VGLL1', 'rs_KCNC2', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KRT81', 'rs_EEF1A2', 'rs_SERPINA11', 'rs_MMP1', 'rs_KRT16', 'rs_MSLN', 'rs_A2ML1', 'rs_SLC6A14', 'rs_PTPRT', 'rs_CA9', 'rs_CYP4Z2P', 'rs_KLHDC7A', 'rs_C10orf82', 'rs_ESR1', 'rs_PDZK1', 'rs_SYT9', 'rs_GRPR', 'rs_ROPN1', 'rs_ANKRD30B', 'rs_NBPF4', 'rs_CHAD', 'rs_WNK4', 'rs_CST5', 'rs_ABCC8', 'rs_SORCS1', 'rs_NKAIN1', 'rs_SLC44A4', 'rs_LY6D', 'rs_CYP4F22', 'rs_GLYATL2', 'rs_BBOX1', 'rs_CAPN8', 'rs_LCN2', 'rs_PGLYRP2', 'rs_PI3', 'rs_NBPF6', 'rs_SCUBE2', 'rs_RIMS4', 'rs_ROPN1B', 'rs_IL20', 'rs_NEK10', 'rs_S100A8', 'rs_TRIM29'

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold: 3
Train shape: (439, 1687)
Test shape: (110, 1687)

Selected Features:
['rs_CPB1', 'rs_TFF1', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_SERPINA6', 'rs_AGR3', 'rs_DHRS2', 'rs_KCN

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold: 4
Train shape: (439, 1687)
Test shape: (110, 1687)

Selected Features:
['rs_CPB1', 'rs_TFF1', 'rs_HMGCS2', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_SERPINA6', 'rs_AGR3', 'rs_DHRS2', 'rs_KCNJ3', 'rs_C4orf7', 'rs_GABRP', 'rs_STAC2', 'rs_CEACAM5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_KRT6B', 'rs_CST9', 'rs_KLK5', 'rs_TFF3', 'rs_BMPR1B', 'rs_SLC34A2', 'rs_AGR2', 'rs_ABCC11', 'rs_C20orf114', 'rs_KIF1A', 'rs_PROM1', 'rs_ELF5', 'rs_GFRA1', 'rs_PGR', 'rs_VGLL1', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KLK7', 'rs_KRT81', 'rs_EEF1A2', 'rs_SE

...

[LightGBM] [Info] Number of positive: 331, number of negative: 108
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002621 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 31102
[LightGBM] [Info] Number of data points in the train set: 439, number of used features: 300
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Selected Features:
['rs_CPB1', 'rs_SCGB1D2', 'rs_TFF1', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_SERPINA6', 'rs_AGR3', 'rs_KCNJ3', 'rs_C4orf7', 'rs_GABRP', 'rs_SLC30A8', 'rs_STAC2', 'rs_VSTM2A', 'rs_CALML5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_CST9', 'rs_TFF3', 'rs_BMPR1B', 'rs_SLC34A2', 'rs_AGR2', 'rs_NPY1R', 'rs_C20orf114', 'rs_KIF1A', 'rs_LRP2', 'rs_PROM1', 'rs_ELF5', 'rs_GFRA1', 'rs_CYP2A6', 'rs_PGR', 'rs_VGLL1', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KRT81', 'rs_EEF1A2', 'rs_SERPINA11', 'rs_KRT16', 'rs_MSLN', 'rs_A2ML1', 'rs_SLC6A14', 'rs_PTPRT', 'rs_CA9', 'rs_SFRP1', 'rs_CYP4Z2P', 'rs_KLHDC7A', 'rs_C10orf82', 'rs_ESR1', 'rs_PDZK1', 'rs_SYT9', 'rs_GRPR', 'rs_MIA', 'rs_ROPN1', 'rs_ANKRD30B', 'rs_NBPF4', 'rs_CHAD', 'rs_ATP13A5', 'rs_WNK4', 'rs_CST5', 'rs_CLIC6', 'rs_ABCC8', 'rs_SORCS1', 'rs_NKAIN1', 'rs_SLC44A4', 'rs_GSTA1', 'rs_LY6D', 'rs_CYP4F22', 'rs_GLYATL2', 'rs_BBOX1', 'rs_CAPN8', 'rs_LCN2', 'rs_PGLYRP2', 'rs_PI3', 'rs_NBPF6', 'rs_SCUBE2', 'rs_KLK8', 'rs_RIMS4', 'rs_

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold: 1
Train shape: (439, 1687)
Test shape: (110, 1687)

Selected Features:
['rs_CPB1', 'rs_TFF1', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_SERPINA6', 'rs_AGR3', 'rs_DHRS2', 'rs_KCNJ3', 'rs_GABRP', 'rs_SLC30A8', 'rs_STAC2', 'rs_CEACAM5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_CST9', 'rs_TFF3', 'rs_BMPR1B', 'rs_AGR2', 'rs_NPY1R', 'rs_C20orf114', 'rs_KIF1A', 'rs_PROM1', 'rs_ELF5', 'rs_GFRA1', 'rs_PGR', 'rs_VGLL1', 'rs_KCNC2', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KRT81', 'rs_EEF1A2', 'rs_SERPINA11', 'rs_MMP1', 'rs_KRT16', 'rs_MSLN', 'rs_A2ML1', 'rs_SLC6A14', 'rs_PTPRT', 'rs_CA9', 'rs_CYP4Z2P', 'rs_KLHDC7A', 'rs_C10orf82', 'rs_ESR1', 'rs_PDZK1', 'rs_SYT9', 'rs_GRPR', 'rs_ROPN1', 'rs_ANKRD30B', 'rs_NBPF4', 'rs_CHAD', 'rs_WNK4', 'rs_CST5', 'rs_ABCC8', 'rs_SORCS1', 'rs_NKAIN1', 'rs_SLC44A4', 'rs_LY6D', 'rs_CYP4F22', 'rs_GLYATL2', 'rs_BBOX1', 'rs_CAPN8', 'rs_LCN2', 'rs_PGLYRP2', 'rs_PI3', 'rs_NBPF6', 'rs_SCUBE2', 'rs_RIMS4', 'rs_ROPN1B', 'rs_IL20', 'rs_NEK10', 'rs_S100A8', 'rs_TRIM29'

In [ ]:
print(rf_cv)

In [ ]:
xgb_cv

In [ ]:
lgbm_cv

In [ ]:
cat_cv

In [ ]:
svm_cv

In [ ]:
lr_cv

6. Is multi-omics better than single-omics?

Scientifically:

YES — slightly to moderately better overall.

BUT:

not dramatically better.

That is actually realistic.


7. What I think is happening biologically

RNA modality already contains a VERY strong ER-status signal.

Protein also contains strong signal.

CNV contains weaker but meaningful signal.

So:
RNA alone already solves much of the task.

That means multi-omics improvement will likely be:

incremental

not revolutionary.

That is biologically believable.


10. Is the multi-omics setup “more reliable” than single-omics?

My honest interpretation:

Setup	Reliability
RNA only	highest robustness
Protein only	strong
CNV only	moderate
Multi-omics	strongest performance but slightly higher overfitting risk

multi-omics appears stronger,
but slightly less inherently stable than RNA-only